# High-frequency non-potential game: DTB versus RK4

[Open in Colab](https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Ver3/notebooks/oscillatory_nonpotential_frequency_sweep.ipynb)

This notebook tests whether DTB remains accurate as the spatial frequency of a two-player game increases. The payoffs are

\[
\Pi_1(x_1,x_2)=-\frac{\kappa}{2}x_1^2+A x_1\sin(\omega x_2),\qquad
\Pi_2(x_1,x_2)=-\frac{\kappa}{2}x_2^2-A x_2\sin(\omega x_1),
\]

and the pseudo-gradient dynamics are

\[
\dot x=b_\omega(x)=
\begin{pmatrix}
-\kappa x_1+A\sin(\omega x_2)\\
-\kappa x_2-A\sin(\omega x_1)
\end{pmatrix}.
\]

The field is generally non-potential because
\(\partial_{x_2}b_1=A\omega\cos(\omega x_2)\) and
\(\partial_{x_1}b_2=-A\omega\cos(\omega x_1)\) are not equal. We sweep
\(\omega\in\{\pi,4\pi,8\pi,16\pi\}\), changing only the frequency.

The supplied example does not assign numerical values to \(A\) or \(\kappa\), so both are controls below and default to the normalized choice \(A=\kappa=1\). The reference is fourth-order Runge--Kutta (RK4), optionally using substeps smaller than the DTB step. There is no scalar-potential diagnostic for this non-potential game.


In [ ]:
from pathlib import Path
import csv
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Find a checkout locally or on PACE. In Colab, clone the same branch.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in candidates if (path / 'DTB_Ver3').is_dir()), None)
if repo_root is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run inside the repository or upload the DTB_Ver3 folder.')
    repo_root = Path('/content/dtb-colab-experiments')
    if not repo_root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo_root),
        ], check=True)
sys.path.insert(0, str(repo_root.resolve()))

from DTB_Ver3 import (
    ExperimentConfig,
    OscillatoryNonpotentialGame,
    ResidualMLP,
    ResidualMMNN,
    run_experiment,
)

package_dir = repo_root / 'DTB_Ver3'
output_root = package_dir / 'results' / 'oscillatory_nonpotential_frequency_sweep'
output_root.mkdir(parents=True, exist_ok=True)
print('repository:', repo_root)
print('output:', output_root)


## Matched experiment controls

Every frequency uses the same particle labels, neural parameters, tangent-basis rule, DTB step size, and final time. The default residual MMNN is initialized as the identity. DTB advances the accumulated state directly,

\[
X_{k+1}=X_k+hJ_{\theta_k}(z)\alpha_k,
\qquad
\theta_{k+1}=\theta_k+h\alpha_k,
\]

and never replaces \(X_{k+1}\) by a neural-map evaluation. Set `TANGENT_BASIS_SIZE=None` for the full trainable basis. If an integer is used, `SUBSET_TANGENT_SELECTION` controls whether that coordinate subset is fixed or redrawn at every step.

RK4 uses `REFERENCE_STEP_SIZE` as its maximum substep. With the defaults it takes four RK4 substeps per DTB step, which makes its error much smaller than the DTB time-discretization error for this comparison.


In [ ]:
SEED = 2026
KAPPA = 1.0
AMPLITUDE = 1.0
FREQUENCY_MULTIPLES = (1, 4, 8, 16)
OMEGAS = tuple(multiplier * np.pi for multiplier in FREQUENCY_MULTIPLES)

N_PARTICLES = 1000
T_FINAL = 1.0
DTB_STEP_SIZE = 0.001
REFERENCE_STEP_SIZE = 0.00025
SNAPSHOT_TIMES = (0.0, 0.25, 0.5, 0.75, 1.0)

NETWORK_FAMILY = 'mmnn'  # choose 'mmnn' or 'mlp'
MMNN_WIDTH, MMNN_RANK, MMNN_DEPTH = 12, 12, 3
MLP_WIDTH, MLP_DEPTH = 16, 2
ACTIVATION = 'tanh'

TANGENT_BASIS_SIZE = None  # None means the full trainable tangent basis
SUBSET_TANGENT_SELECTION = 'fixed'  # or 'resample_each_step'
SVD_RTOL = 1e-3
JACOBIAN_CHUNK = 256
DEVICE = 'auto'
DTYPE = 'float32'

torch.manual_seed(SEED + 100)
if NETWORK_FAMILY == 'mmnn':
    model_kind = 'residual_mmnn'
    model = ResidualMMNN(
        2,
        width=MMNN_WIDTH,
        rank=MMNN_RANK,
        depth=MMNN_DEPTH,
        activation=ACTIVATION,
        dtype=torch.float32,
        zero_init_output=True,
    )
    network_config = dict(width=MMNN_WIDTH, rank=MMNN_RANK, depth=MMNN_DEPTH)
elif NETWORK_FAMILY == 'mlp':
    model_kind = 'residual_mlp'
    model = ResidualMLP(
        2,
        width=MLP_WIDTH,
        depth=MLP_DEPTH,
        activation=ACTIVATION,
        dtype=torch.float32,
        zero_init_output=True,
    )
    network_config = dict(width=MLP_WIDTH, rank=1, depth=MLP_DEPTH)
else:
    raise ValueError("NETWORK_FAMILY must be 'mmnn' or 'mlp'.")

trainable_parameter_count = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
basis_size = (
    trainable_parameter_count
    if TANGENT_BASIS_SIZE is None
    else min(int(TANGENT_BASIS_SIZE), trainable_parameter_count)
)
print({
    'model': model_kind,
    'trainable_parameters': trainable_parameter_count,
    'tangent_basis_size': basis_size,
    'frequencies': [f'{m}π' for m in FREQUENCY_MULTIPLES],
    'dtb_step': DTB_STEP_SIZE,
    'rk4_max_step': REFERENCE_STEP_SIZE,
})


## Run the frequency sweep

The reported trajectory error is the paired-particle RMS

\[
E_{\mathrm{traj}}(t_k)=
\left(\frac1N\sum_{i=1}^N
\|x^{k}_{i,\mathrm{DTB}}-x^{k}_{i,\mathrm{RK4}}\|_2^2\right)^{1/2}.
\]

The relative final error divides this quantity by the RMS norm of the final RK4 cloud. The tangent diagnostic is already relative:

\[
r^k_{\mathrm{proj}}=
\frac{\|J^k\alpha^k-b_\omega(X_k)\|_2}
{\|b_\omega(X_k)\|_2}.
\]


In [ ]:
runs = {}
records = []

for multiplier, omega in zip(FREQUENCY_MULTIPLES, OMEGAS):
    print(f'\n=== omega = {multiplier} pi ===', flush=True)
    game = OscillatoryNonpotentialGame(
        kappa=KAPPA,
        amplitude=AMPLITUDE,
        omega=float(omega),
    )
    config = ExperimentConfig(
        dynamics='deterministic',
        run_reference=True,
        reference_integrator='rk4',
        reference_step_size=REFERENCE_STEP_SIZE,
        particle_count=N_PARTICLES,
        initial_law='uniform',
        initial_low=-1.0,
        initial_high=1.0,
        step_size=DTB_STEP_SIZE,
        final_time=T_FINAL,
        snapshot_times=SNAPSHOT_TIMES,
        activation=ACTIVATION,
        model_kind=model_kind,
        zero_init_output=True,
        basis_size=basis_size,
        subset_tangent_selection=SUBSET_TANGENT_SELECTION,
        tangent_input_mode='fixed_initial_labels',
        track_network_map=False,
        svd_rtol=SVD_RTOL,
        jacobian_chunk=JACOBIAN_CHUNK,
        seed=SEED,
        dtype=DTYPE,
        device=DEVICE,
        progress_reports=5,
        output_dir=output_root / f'omega_{multiplier}pi',
        **network_config,
    )
    result = run_experiment(game, config, model=model)
    runs[multiplier] = result

    reference_scale = np.sqrt(np.mean(np.sum(result.reference_final_particles**2, axis=1)))
    relative_final_rms = result.final_paired_rms / max(reference_scale, np.finfo(float).tiny)
    records.append([
        float(multiplier),
        float(omega),
        float(result.final_paired_rms),
        float(relative_final_rms),
        float(result.relative_projection_error.mean()),
        float(result.relative_projection_error[-1]),
        float(result.relative_projection_error.max()),
        float(result.elapsed_seconds),
    ])

columns = (
    'omega_over_pi', 'omega', 'final_paired_rms', 'relative_final_rms',
    'mean_relative_projection_error', 'final_relative_projection_error',
    'max_relative_projection_error', 'elapsed_seconds',
)
records = np.asarray(records)
summary_path = output_root / 'frequency_sweep.csv'
np.savetxt(
    summary_path,
    records,
    delimiter=',',
    header=','.join(columns),
    comments='',
)

print('\nFrequency-sweep summary')
print('omega/pi | final RMS | relative final RMS | mean projection residual')
for row in records:
    print(f'{row[0]:8g} | {row[2]:9.3e} | {row[3]:18.3e} | {row[4]:24.3e}')
print('saved:', summary_path)


## Accuracy versus spatial frequency

The left plots answer the main question directly: accuracy is retained when the error curves remain low as `omega/pi` grows. The right plots help separate causes. A growing projection residual means the neural tangent space cannot represent the high-frequency velocity well; a low projection residual with a growing RK4 discrepancy points instead to accumulated time-stepping or parameter-evolution error.


In [ ]:
frequency_axis = records[:, 0]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].loglog(frequency_axis, records[:, 2], 'o-', linewidth=2)
axes[0, 0].set(
    xlabel=r'$\omega/\pi$', ylabel='final paired RMS',
    title='DTB versus refined RK4 at final time',
)

axes[0, 1].loglog(frequency_axis, records[:, 3], 'o-', linewidth=2, color='tab:orange')
axes[0, 1].set(
    xlabel=r'$\omega/\pi$', ylabel='relative final RMS',
    title='Final error / RK4 cloud RMS norm',
)

for multiplier, result in runs.items():
    axes[1, 0].semilogy(result.times, np.maximum(result.trajectory_rms_error, 1e-12),
                       label=fr'$\omega={multiplier}\pi$')
    axes[1, 1].semilogy(
        result.projection_times,
        np.maximum(result.relative_projection_error, 1e-12),
        label=fr'$\omega={multiplier}\pi$',
    )
axes[1, 0].set(xlabel='time', ylabel='paired RMS', title='Trajectory error')
axes[1, 1].set(xlabel='time', ylabel='relative residual', title='Tangent projection error')
axes[1, 0].legend()
axes[1, 1].legend()
for axis in axes.flat:
    axis.grid(True, alpha=0.3)

accuracy_path = output_root / 'accuracy_vs_frequency.png'
fig.savefig(accuracy_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', accuracy_path)


## Vector fields and final clouds

Each column changes only the spatial frequency. The upper row shows the game field; the lower rows compare the final accumulated DTB particles with the refined RK4 particles from the same initial labels.


In [ ]:
grid = np.linspace(-1.1, 1.1, 81)
grid_x, grid_y = np.meshgrid(grid, grid)

fig, axes = plt.subplots(3, len(FREQUENCY_MULTIPLES), figsize=(16, 11), constrained_layout=True)
for column, (multiplier, omega) in enumerate(zip(FREQUENCY_MULTIPLES, OMEGAS)):
    field_x = -KAPPA * grid_x + AMPLITUDE * np.sin(omega * grid_y)
    field_y = -KAPPA * grid_y - AMPLITUDE * np.sin(omega * grid_x)
    speed = np.sqrt(field_x**2 + field_y**2)
    axes[0, column].streamplot(
        grid_x, grid_y, field_x, field_y,
        color=speed, cmap='viridis', density=1.1, linewidth=0.7,
    )
    axes[0, column].set_title(fr'$\omega={multiplier}\pi$')

    result = runs[multiplier]
    dtb = result.dtb_final_particles
    reference = result.reference_final_particles
    axes[1, column].scatter(dtb[:, 0], dtb[:, 1], s=5, alpha=0.35, color='tab:orange')
    axes[2, column].scatter(reference[:, 0], reference[:, 1], s=5, alpha=0.35, color='tab:blue')
    for row in range(3):
        axes[row, column].set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), aspect='equal')
        axes[row, column].grid(True, alpha=0.15)

axes[0, 0].set_ylabel('vector field')
axes[1, 0].set_ylabel('DTB final')
axes[2, 0].set_ylabel('RK4 final')
for axis in axes[2, :]:
    axis.set_xlabel(r'$x_1$')

cloud_path = output_root / 'final_clouds_vs_frequency.png'
fig.savefig(cloud_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', cloud_path)


## Hardest-frequency diagnostics

This view focuses on `omega = 16 pi`. Paired DTB/RK4 snapshots expose when the discrepancy begins. The remaining panels show the tangent representation error, the accumulated state error, the coefficient norm, and the retained tangent rank.


In [ ]:
hardest = runs[max(FREQUENCY_MULTIPLES)]
snapshot_times = sorted(hardest.dtb_snapshots)
fig, axes = plt.subplots(2, len(snapshot_times), figsize=(3.3 * len(snapshot_times), 6.5), constrained_layout=True)
for column, time_value in enumerate(snapshot_times):
    dtb = hardest.dtb_snapshots[time_value]
    reference = hardest.reference_snapshots[time_value]
    axes[0, column].scatter(reference[:, 0], reference[:, 1], s=5, alpha=0.28,
                            color='tab:blue', label='RK4')
    axes[0, column].scatter(dtb[:, 0], dtb[:, 1], s=5, alpha=0.28,
                            color='tab:orange', label='DTB')
    axes[0, column].set(title=f't={time_value:g}', xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), aspect='equal')
    paired_distance = np.linalg.norm(dtb - reference, axis=1)
    axes[1, column].hist(paired_distance, bins=35, color='tab:purple', alpha=0.8)
    axes[1, column].set(xlabel='paired distance', ylabel='count')
axes[0, 0].legend(loc='upper right')
hardest_cloud_path = output_root / 'omega_16pi_snapshot_comparison.png'
fig.savefig(hardest_cloud_path, dpi=180, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
axes[0, 0].semilogy(hardest.projection_times, np.maximum(hardest.relative_projection_error, 1e-12))
axes[0, 0].set(xlabel='time', ylabel='relative residual', title='Tangent projection')
axes[0, 1].semilogy(hardest.times, np.maximum(hardest.trajectory_rms_error, 1e-12))
axes[0, 1].set(xlabel='time', ylabel='paired RMS', title='DTB versus RK4')
axes[1, 0].plot(hardest.projection_times, hardest.alpha_norm)
axes[1, 0].set(xlabel='time', ylabel=r'$\|\alpha_k\|_2$', title='Coefficient norm')
axes[1, 1].plot(hardest.projection_times, hardest.retained_rank)
axes[1, 1].set(xlabel='time', ylabel='rank', title='Retained tangent rank')
for axis in axes.flat:
    axis.grid(True, alpha=0.3)
hardest_diagnostic_path = output_root / 'omega_16pi_diagnostics.png'
fig.savefig(hardest_diagnostic_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', hardest_cloud_path)
print('saved:', hardest_diagnostic_path)


## Package the numerical outputs

The archive contains the frequency table, all arrays saved by each DTB/RK4 run, and the figures. Download this zip through JupyterLab's file browser on PACE.


In [ ]:
archive_path = Path(shutil.make_archive(
    str(output_root),
    'zip',
    root_dir=output_root,
))
print('result archive:', archive_path)
